# Problem 74: Cascading Failure Isolation

**Difficulty:** Hard | **Framework:** LangGraph

**Categories:** error-recovery, multi-agent

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/cascading-failure-isolation).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- Each agent must run in its own isolated error boundary
- If one agent crashes, the others must continue executing
- The final output must include results from all successful agents and a failure note for crashed ones
- The pipeline must not re-raise exceptions from individual agent failures


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    topic: str
    research: str
    analysis: str
    visualization: str
    report: str

def research_agent(state: State) -> State:
    return {"research": f"Research findings on {state['topic']}: [detailed research]"}

def analysis_agent(state: State) -> State:
    # BUG: This agent crashes and takes down the entire pipeline
    raise RuntimeError("Analysis model failed to load")

def visualization_agent(state: State) -> State:
    return {"visualization": f"Charts generated for {state['topic']}"}

def report_agent(state: State) -> State:
    return {"report": f"Report: Research={state['research']}, Analysis={state['analysis']}, Viz={state['visualization']}"}

graph = StateGraph(State)
graph.add_node("research", research_agent)
graph.add_node("analysis", analysis_agent)
graph.add_node("visualization", visualization_agent)
graph.add_node("report", report_agent)

graph.add_edge(START, "research")
graph.add_edge("research", "analysis")
graph.add_edge("analysis", "visualization")
graph.add_edge("visualization", "report")
graph.add_edge("report", END)

app = graph.compile()

# Test: One agent crash kills the entire pipeline
result = app.invoke({"topic": "AI trends 2025", "research": "", "analysis": "", "visualization": "", "report": ""})
print(result["report"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Wrap each agent's execution in a try/except block that captures the error and stores a fallback result
2. Run agents in parallel if possible, or sequentially with isolated error handling per step
3. Aggregate results at the end: include successful outputs and clearly mark which agents failed


## 7. Evaluation Criteria

Check your solution against these criteria:

- Failure is isolated to one agent
- Other agents continue running
- Partial results returned
- Failed agent is reported
